In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd, log, ceil

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate, QFT
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


# generalised modular multiplication
def build_mod_mult_unitary(k, N, n_bits):
    dim = 2**n_bits
    U = np.zeros((dim, dim))
    for x in range(dim):
        if x == 0:
            U[0][0] = 1  # zero always maps to zero
        elif x < N and gcd(x, N) == 1:
            # normal case - do the modular multiplication
            y = (x * k) % N
            U[y][x] = 1
        else:
            # x is outside the valid range (>= N), just act as identity
            U[x][x] = 1
    return U


# shor's algorithm following IBM's tutorial
def build_shor_circuit(N, a, num_control, num_target):

    # named registers so the circuit diagram is actually readable
    control = QuantumRegister(num_control, name='Counting')
    target  = QuantumRegister(num_target,  name='Target')
    output  = ClassicalRegister(num_control, name='output')
    circuit = QuantumCircuit(control, target, output)

    # need to start the target register in |1> for the modular exponentiation to work
    circuit.x(num_control)

    # put all counting qubits into equal superposition
    for k in range(num_control):
        circuit.h(k)
    circuit.barrier()

    # the core of the algorithm: controlled modular exponentiation
    # each counting qubit k controls multiplication by a^(2^k) mod N
    n_mod_mults = 0
    for k in range(num_control):
        power = pow(a, 2**k, N)
        if power == 1:
            continue  # multiplying by 1 is a no-op, skip it

        # build the unitary for this specific multiplication
        U      = build_mod_mult_unitary(power, N, num_target)
        gate   = UnitaryGate(U)
        gate.name = f'M_{power}'

        # make it controlled by counting qubit k
        c_gate = gate.control()
        circuit.compose(c_gate,
                        qubits=[k] + list(range(num_control, num_control + num_target)),
                        inplace=True)
        n_mod_mults += 1

    circuit.barrier()

    # inverse QFT converts the phase kickback into something we can actually measure
    circuit.compose(QFT(num_control, inverse=True), qubits=control, inplace=True)
    circuit.barrier()

    # finally, measure the counting register to get our phase estimate
    circuit.measure(control, output)

    return circuit, n_mod_mults


print('Imports and circuit builders loaded.')

In [ ]:
# connect to IBM Quantum (only need to do this once)
# first time only - paste your API token and CRN, then comment out after saving
QiskitRuntimeService.save_account(
    channel='ibm_quantum_platform',
    token='token here',
    instance='instance here',
    overwrite=True
)

service = QiskitRuntimeService(channel='ibm_quantum_platform')
backend = service.backend('ibm_fez')

print(f'Connected to:     {backend.name}')
print(f'Qubits available: {backend.num_qubits}')
print(f'Processor:        {backend.processor_type}')

In [ ]:
# build both circuits
a = 2

# N=15: 4 target qubits to represent numbers up to 15, 8 counting qubits for precision
N15          = 15
nt15         = ceil(log(N15, 2))
nc15         = 2 * nt15
circuit15, n_mults15 = build_shor_circuit(N15, a, nc15, nt15)

# N=21: needs 5 target qubits and 10 counting qubits (bigger circuit)
N21          = 21
nt21         = ceil(log(N21, 2))
nc21         = 2 * nt21
circuit21, n_mults21 = build_shor_circuit(N21, a, nc21, nt21)

print(f'N=15: {nc15} counting + {nt15} target = {nc15+nt15} qubits, {n_mults15} controlled multiplications')
print(f'N=21: {nc21} counting + {nt21} target = {nc21+nt21} qubits, {n_mults21} controlled multiplications')

In [ ]:
# transpile both circuits for the real device
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

compiled15 = pm.run(circuit15)
compiled21 = pm.run(circuit21)

gates15 = sum(compiled15.count_ops().values())
gates21 = sum(compiled21.count_ops().values())

print(f'N=15  →  Total gates: {gates15:,}   Depth: {compiled15.depth()}')
print(f'N=21  →  Total gates: {gates21:,}   Depth: {compiled21.depth()}')

In [ ]:
# submit both jobs back-to-back before waiting for results
# save both job IDs in case the session disconnects - retrieve with:

shots   = 4096
sampler = SamplerV2(backend)

job15 = sampler.run([compiled15], shots=shots)
job21 = sampler.run([compiled21], shots=shots)

print(f'Both jobs submitted!')
print(f'N=15 Job ID: {job15.job_id()}')
print(f'N=21 Job ID: {job21.job_id()}')
print(f'Shots: {shots} each')

In [ ]:
# retrieve both results
result15 = job15.result()
result21 = job21.result()

counts15 = result15[0].data.output.get_counts()
counts21 = result21[0].data.output.get_counts()

print(f'N=15 complete:  {len(counts15)} unique outcomes  |  top: {max(counts15.values())} shots ({100*max(counts15.values())/shots:.1f}%)')
print(f'N=21 complete:  {len(counts21)} unique outcomes  |  top: {max(counts21.values())} shots ({100*max(counts21.values())/shots:.1f}%)')

In [ ]:
# plot histograms for N=15 and N=21 (separate figures)
import matplotlib.patches as mpatches

def plot_clean_histogram(counts, shots, expected_peaks, title, filename=None):
    # show top 30 bars by count, always include expected peaks
    # filters out very low counts to keep the plot readable
    decimal_counts = {int(k, 2): v for k, v in counts.items()}

    # always include expected peaks; fill remaining slots with highest counts
    top_keys = set(k for k, v in sorted(decimal_counts.items(),
                                         key=lambda x: -x[1])[:30])
    top_keys |= set(expected_peaks)
    top = {k: decimal_counts.get(k, 0) for k in sorted(top_keys) if decimal_counts.get(k, 0) > 0}

    keys   = list(top.keys())
    vals   = list(top.values())
    colors = ['crimson' if k in expected_peaks else 'steelblue' for k in keys]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(range(len(keys)), vals, color=colors, edgecolor='white', width=0.7)

    # put the actual count on top of each bar
    for bar, count in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(count), ha='center', va='bottom', fontsize=9)

    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels([str(k) for k in keys], rotation=45, ha='right', fontsize=10)
    ax.set_xlabel('Measured Value (decimal)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    peak_str = ', '.join(str(p) for p in sorted(expected_peaks))
    ax.legend(handles=[
        mpatches.Patch(color='crimson',   label=f'Expected peaks ({peak_str})'),
        mpatches.Patch(color='steelblue', label='Noise (other measurements)'),
    ], fontsize=11)
    ax.set_ylim(0, max(vals) + 3)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    if filename:
        fig.savefig(filename, dpi=200, bbox_inches='tight', facecolor='white')
        print(f'Saved: {filename}')
    plt.show()

# N=15 - should see peaks at multiples of 256/4 = 64
plot_clean_histogram(
    counts15, shots,
    expected_peaks={0, 64, 128, 192},
    title=f"Shor's Algorithm on IBM Hardware (ibm_fez): N=15, a=2\n{shots} shots — Expected peaks: {{0, 64, 128, 192}}"
)

# N=21 - expected peaks at multiples of 1024/6
plot_clean_histogram(
    counts21, shots,
    expected_peaks={0, 171, 341, 512, 683, 853},
    title=f"Shor's Algorithm on IBM Hardware (ibm_fez): N=21, a=2\n{shots} shots — Expected peaks: {{0, 171, 341, 512, 683, 853}}"
)

In [ ]:
# extract the phase from each measurement outcome for N=15
# the measured integer m gives us a phase estimate of m / 2^n

rows_phase15 = []
for bitstring in sorted(counts15, key=lambda x: counts15[x], reverse=True):
    count   = counts15[bitstring]
    decimal = int(bitstring, 2)
    phase   = decimal / (2**nc15)
    rows_phase15.append([
        bitstring, decimal, count,
        f'{count/shots:.1%}',
        f'{decimal}/{2**nc15} = {phase:.4f}'
    ])

print('Phase extraction for N=15, a=2:')
df_phase15 = pd.DataFrame(rows_phase15[:15],
    columns=['Bitstring', 'Decimal', 'Count', 'Probability', 'Phase'])
print(df_phase15.to_string(index=False))

In [ ]:
# same phase extraction but for N=21

rows_phase21 = []
for bitstring in sorted(counts21, key=lambda x: counts21[x], reverse=True):
    count   = counts21[bitstring]
    decimal = int(bitstring, 2)
    phase   = decimal / (2**nc21)
    rows_phase21.append([
        bitstring, decimal, count,
        f'{count/shots:.1%}',
        f'{decimal}/{2**nc21} = {phase:.4f}'
    ])

print('Phase extraction for N=21, a=2:')
df_phase21 = pd.DataFrame(rows_phase21[:15],
    columns=['Bitstring', 'Decimal', 'Count', 'Probability', 'Phase'])
print(df_phase21.to_string(index=False))

In [ ]:
# use continued fractions to turn the phase into a period guess for N=15,
# then try to actually factor N with it

def extract_periods(counts, N, a, num_control, shots):
    # iterate all outcomes, return the table and how many shots gave factors
    rows, success_shots = [], 0
    for bitstring in sorted(counts, key=lambda x: counts[x], reverse=True):
        count   = counts[bitstring]
        decimal = int(bitstring, 2)
        phase   = decimal / (2**num_control)

        # continued fraction approximation
        frac    = Fraction(phase).limit_denominator(N)
        r       = frac.denominator

        # check if this is actually a valid period
        valid   = r > 1 and pow(a, r, N) == 1
        factors = ''
        if valid and r % 2 == 0:
            x = pow(a, r // 2, N)
            if x != N - 1:  # trivial case, useless factors
                f1, f2 = gcd(x - 1, N), gcd(x + 1, N)
                if 1 < f1 < N:
                    factors = f'{f1} x {N // f1}'
                    success_shots += count
                elif 1 < f2 < N:
                    factors = f'{f2} x {N // f2}'
                    success_shots += count
        rows.append([f'{phase:.4f}', f'{frac.numerator}/{frac.denominator}',
                     r, 'Yes' if valid else 'No', count,
                     factors if factors else '-'])
    df = pd.DataFrame(rows, columns=['Phase','Fraction','Guess for r','Valid','Count','Factors'])
    return df, success_shots

df15, success15 = extract_periods(counts15, N15, a, nc15, shots)

# split into successful and failed outcomes so we can see both
successful15 = df15[df15['Factors'] != '-'].sort_values('Count', ascending=False)
failed15     = df15[df15['Factors'] == '-'].sort_values('Count', ascending=False).head(10)

print(f'Period extraction for N=15, a=2:')
print(f'\n--- Successful outcomes ({len(successful15)} rows, {success15} shots total) ---')
print(successful15.to_string(index=False))
print(f'\n--- Top 10 unsuccessful outcomes (for comparison) ---')
print(failed15.to_string(index=False))
print(f'\nSuccess rate: {100*success15/shots:.1f}%  ({success15}/{shots} shots)')
if success15 > 0:
    print(f'Factors found: 3 x 5 = 15  \u2713')
    print(f'Avg shots to first success: {round(shots/success15)}')

In [ ]:
# same thing but for N=21 now

df21, success21 = extract_periods(counts21, N21, a, nc21, shots)

successful21 = df21[df21['Factors'] != '-'].sort_values('Count', ascending=False)
failed21     = df21[df21['Factors'] == '-'].sort_values('Count', ascending=False).head(10)

print(f'Period extraction for N=21, a=2:')
print(f'\n--- Successful outcomes ({len(successful21)} rows, {success21} shots total) ---')
print(successful21.to_string(index=False))
print(f'\n--- Top 10 unsuccessful outcomes (for comparison) ---')
print(failed21.to_string(index=False))
print(f'\nSuccess rate: {100*success21/shots:.1f}%  ({success21}/{shots} shots)')
if success21 > 0:
    print(f'Factors found: 3 x 7 = 21  \u2713')
    print(f'Avg shots to first success: {round(shots/success21)}')
else:
    print('No successful factorisation.')
    print('Circuit depth far exceeds hardware coherence limits.')

In [ ]:
# theoretical random-chance success rate for the continued fractions algorithm
# check every possible measurement outcome and see how many give correct factors
from fractions import Fraction
from math import gcd

def random_success_rate(N, a, num_control):
    possible  = 2**num_control
    lucky = sum(
        1 for d in range(possible)
        if (lambda r: r > 1 and pow(a, r, N) == 1 and r % 2 == 0 and
            (lambda x: x != N-1 and
             (1 < gcd(x-1, N) < N or 1 < gcd(x+1, N) < N)
            )(pow(a, r//2, N))
           )(Fraction(d/possible).limit_denominator(N).denominator)
    )
    return lucky, possible, 100 * lucky / possible

lucky15, pos15, pct15 = random_success_rate(N=15, a=2, num_control=8)
lucky21, pos21, pct21 = random_success_rate(N=21, a=2, num_control=10)

print(f'N=15: {lucky15}/{pos15} outcomes give correct factors = {pct15:.2f}% by random chance')
print(f'N=21: {lucky21}/{pos21} outcomes give correct factors = {pct21:.2f}% by random chance')